<a href="https://colab.research.google.com/github/Rakshit-Gupta-77/Machine-Learning-Journey/blob/main/column_transformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd

In [2]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

In [3]:
df = pd.read_csv('covid_toy.csv')
df.head()

,age,gender,fever,cough,city,has_covid
0,60,Male,103.0,Mild,Kolkata,No
1,27,Male,100.0,Mild,Delhi,Yes
2,42,Male,101.0,Mild,Delhi,No
3,31,Female,98.0,Mild,Kolkata,No
4,65,Female,101.0,Mild,Mumbai,No


## Understand the dataset

The notebook expects these feature columns: `age`, `gender`, `fever`, `cough`, and `city`, with `has_covid` as the target.

- `age` → numerical
- `fever` → numerical but may contain missing values
- `cough` → ordinal categorical (`Mild < Strong`)
- `gender`, `city` → nominal categorical
- `has_covid` → target

In [4]:
df.isnull().sum()

,0
age,0
gender,0
fever,10
cough,0
city,0
has_covid,0


In [5]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=['has_covid'])
y = df['has_covid']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print('X_train:', X_train.shape)
print('X_test :', X_test.shape)

X_train: (80, 5)
X_test : (20, 5)


## 1. Aam Zindagi — Manual preprocessing

Here we preprocess each column separately. **Important correction:** fit each transformer only on training data; use `transform()` on the test data. Fitting again on the test set would create a different mapping and can cause data leakage.

In [6]:
# 1) Missing-value imputation for fever
si = SimpleImputer(strategy='mean')

X_train_fever = si.fit_transform(X_train[['fever']])
X_test_fever = si.transform(X_test[['fever']])

print('Train fever shape:', X_train_fever.shape)
print('Test fever shape :', X_test_fever.shape)

Train fever shape: (80, 1)
Test fever shape : (20, 1)


In [7]:
# 2) Ordinal encoding for cough
# The order is meaningful: Mild < Strong
oe = OrdinalEncoder(categories=[['Mild', 'Strong']])

X_train_cough = oe.fit_transform(X_train[['cough']])
X_test_cough = oe.transform(X_test[['cough']])

print('Train cough shape:', X_train_cough.shape)
print('Test cough shape :', X_test_cough.shape)

Train cough shape: (80, 1)
Test cough shape : (20, 1)


In [8]:
# 3) One-hot encoding for nominal categories: gender and city
# drop='first' gives K-1 dummy columns.
# sparse_output=False is the current scikit-learn API.
ohe = OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore')

X_train_gender_city = ohe.fit_transform(X_train[['gender', 'city']])
X_test_gender_city = ohe.transform(X_test[['gender', 'city']])

print('Train encoded shape:', X_train_gender_city.shape)
print('Test encoded shape :', X_test_gender_city.shape)
print('Encoded columns:', ohe.get_feature_names_out(['gender', 'city']))

Train encoded shape: (80, 4)
Test encoded shape : (20, 4)
Encoded columns: ['gender_Male' 'city_Delhi' 'city_Kolkata' 'city_Mumbai']


In [9]:
# 4) Extract the numerical age column
X_train_age = X_train[['age']].to_numpy()
X_test_age = X_test[['age']].to_numpy()

print('Train age shape:', X_train_age.shape)
print('Test age shape :', X_test_age.shape)

Train age shape: (80, 1)
Test age shape : (20, 1)


In [10]:
# 5) Combine all transformed features
X_train_transformed = np.concatenate(
    (X_train_age, X_train_fever, X_train_gender_city, X_train_cough),
    axis=1
)

X_test_transformed = np.concatenate(
    (X_test_age, X_test_fever, X_test_gender_city, X_test_cough),
    axis=1
)

print('Final X_train shape:', X_train_transformed.shape)
print('Final X_test shape :', X_test_transformed.shape)

Final X_train shape: (80, 7)
Final X_test shape : (20, 7)


## 2. Mentos Zindagi — ColumnTransformer

Instead of manually preprocessing every column and concatenating NumPy arrays, `ColumnTransformer` lets us define the complete preprocessing pipeline in one place.

In [11]:
from sklearn.compose import ColumnTransformer

In [12]:
transformer = ColumnTransformer(
    transformers=[
        ('tnf1', SimpleImputer(strategy='mean'), ['fever']),
        ('tnf2', OrdinalEncoder(categories=[['Mild', 'Strong']]), ['cough']),
        ('tnf3', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), ['gender', 'city'])
    ],
    remainder='passthrough'
)

In [13]:
X_train_ct = transformer.fit_transform(X_train)
X_test_ct = transformer.transform(X_test)

print('X_train transformed shape:', X_train_ct.shape)
print('X_test transformed shape :', X_test_ct.shape)

X_train transformed shape: (80, 7)
X_test transformed shape : (20, 7)


## Why ColumnTransformer is better

It combines all preprocessing steps into one object and keeps the **same learned transformations** for train and test data.

### Final flow

`Raw data → split → fit transformers on train → transform train/test → ML model`

## Quick Revision

- `SimpleImputer` → fills missing values.
- `OrdinalEncoder` → ordered categorical feature such as `Mild < Strong`.
- `OneHotEncoder` → nominal categories such as gender/city.
- `fit_transform()` → training data.
- `transform()` → test data.
- `ColumnTransformer` → combines different preprocessing methods for different columns.
- `drop="first"` → K-1 dummy encoding.
- `sparse_output=False` → current scikit-learn way to request a dense array.